In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

## Config

In [ ]:
np.random.seed(42)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

N_USERS = 2_000
N_NONPROFITS = 75
N_CAMPAIGNS = 250
N_DONATIONS = 10_000

START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

In [4]:
def random_dates(n, start=START_DATE, end=END_DATE):
    delta_seconds = int((end - start).total_seconds())
    offsets = np.random.randint(0, delta_seconds, size=n)
    return [start + timedelta(seconds=int(x)) for x in offsets]

## Users

In [5]:
users = pd.DataFrame({
    "user_id": range(1, N_USERS + 1),
    "created_at": random_dates(N_USERS),
    "acquisition_channel": np.random.choice(
        ["organic", "paid_search", "social", "email", "referral"],
        size=N_USERS,
        p=[0.35, 0.20, 0.20, 0.15, 0.10]
    ),
    "country": np.random.choice(
        ["US", "CA", "GB", "AU", "MX"],
        size=N_USERS,
        p=[0.75, 0.08, 0.07, 0.05, 0.05]
    )
})

# Add a few nulls intentionally
users.loc[np.random.choice(users.index, size=25, replace=False), "acquisition_channel"] = None


## Non-Profits

In [6]:
cause_categories = [
    "Education",
    "Health",
    "Environment",
    "Animal Welfare",
    "Human Services",
    "Arts & Culture",
    "Disaster Relief"
]

nonprofits = pd.DataFrame({
    "nonprofit_id": range(1, N_NONPROFITS + 1),
    "nonprofit_name": [f"Nonprofit {i}" for i in range(1, N_NONPROFITS + 1)],
    "cause_category": np.random.choice(cause_categories, size=N_NONPROFITS),
    "created_at": random_dates(N_NONPROFITS, datetime(2024, 1, 1), datetime(2025, 6, 30))
})

## Campaigns

In [7]:
campaign_created_at = random_dates(N_CAMPAIGNS, datetime(2025, 1, 1), datetime(2025, 10, 31))

campaigns = pd.DataFrame({
    "campaign_id": range(1, N_CAMPAIGNS + 1),
    "nonprofit_id": np.random.choice(nonprofits["nonprofit_id"], size=N_CAMPAIGNS),
    "campaign_name": [f"Campaign {i}" for i in range(1, N_CAMPAIGNS + 1)],
    "goal_amount": np.random.choice(
        [1000, 2500, 5000, 10000, 25000, 50000],
        size=N_CAMPAIGNS,
        p=[0.15, 0.25, 0.25, 0.20, 0.10, 0.05]
    ),
    "created_at": campaign_created_at,
    "campaign_type": np.random.choice(
        ["individual", "team", "event", "recurring"],
        size=N_CAMPAIGNS,
        p=[0.35, 0.25, 0.25, 0.15]
    )
})

## Donations

In [8]:
donation_statuses = np.random.choice(
    ["completed", "failed", "refunded"],
    size=N_DONATIONS,
    p=[0.84, 0.10, 0.06]
)

donations = pd.DataFrame({
    "donation_id": range(1, N_DONATIONS + 1),
    "user_id": np.random.choice(users["user_id"], size=N_DONATIONS),
    "campaign_id": np.random.choice(campaigns["campaign_id"], size=N_DONATIONS),
    "amount": np.round(
        np.random.lognormal(mean=3.4, sigma=0.8, size=N_DONATIONS),
        2
    ),
    "status": donation_statuses,
    "created_at": random_dates(N_DONATIONS),
    "is_recurring": np.random.choice([True, False], size=N_DONATIONS, p=[0.18, 0.82])
})

# Cap extreme donations to keep numbers reasonable
donations["amount"] = donations["amount"].clip(5, 2_500)

# Failed donations should not have collected revenue
donations.loc[donations["status"] == "failed", "amount"] = 0

# Add a few intentional raw data issues
donations.loc[np.random.choice(donations.index, size=20, replace=False), "user_id"] = None
donations.loc[np.random.choice(donations.index, size=15, replace=False), "campaign_id"] = None



## Payments

In [9]:
successful_donations = donations[donations["status"].isin(["completed", "refunded"])].copy()

payments = pd.DataFrame({
    "payment_id": range(1, len(successful_donations) + 1),
    "donation_id": successful_donations["donation_id"].values,
    "processor": np.random.choice(
        ["stripe", "paypal", "venmo"],
        size=len(successful_donations),
        p=[0.70, 0.20, 0.10]
    ),
    "payment_status": np.where(
        successful_donations["status"].values == "refunded",
        "refunded",
        "captured"
    ),
    "processed_at": successful_donations["created_at"].values,
    "gross_amount": successful_donations["amount"].values
})

# Simple fee logic: 2.9% + $0.30
payments["fee_amount"] = np.round(payments["gross_amount"] * 0.029 + 0.30, 2)
payments["net_amount"] = np.round(payments["gross_amount"] - payments["fee_amount"], 2)

# Save files

In [10]:
users.to_csv(OUTPUT_DIR / "raw_users.csv", index=False)
nonprofits.to_csv(OUTPUT_DIR / "raw_nonprofits.csv", index=False)
campaigns.to_csv(OUTPUT_DIR / "raw_campaigns.csv", index=False)
donations.to_csv(OUTPUT_DIR / "raw_donations.csv", index=False)
payments.to_csv(OUTPUT_DIR / "raw_payments.csv", index=False)

print("CSV files created:")
for file in OUTPUT_DIR.glob("*.csv"):
    print(f"- {file}")

CSV files created:
- fundraising_raw_data/raw_nonprofits.csv
- fundraising_raw_data/raw_donations.csv
- fundraising_raw_data/raw_payments.csv
- fundraising_raw_data/raw_users.csv
- fundraising_raw_data/raw_campaigns.csv
